In [1]:
#premise_pathways
import bw2data
import bw2io as bi
from datapackage import Package
import premise as pr
from por_supplier_linking import configure_por_export_support, attach_por_supplier_linking
configure_por_export_support()
import bw2data as bd
import pycountry
import re
import pandas as pd

# Now import shared variables
from config import *
from private_keys import *

#Standard vars
bw2data.projects.set_current(PROJECT_NAME)
pr.__version__,PROJECT_NAME

c:\Users\terlouw_t\.conda\envs\premise_pathways_new\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


13:23:33+0200 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.
Applied premise external-market compatibility fix for empty uncertainty bounds.
Applied export guard for scenarios with validated repo-managed PoR supplier links.
Installed repo Pathways mapping compatibility guard for metals diagnostics.
Installed exact inventory supplier compatibility hooks for configured additional inventories.


((2, 5, 2), 'por')

### 1. Here we decide whether we want to generate a new brightway database or export the databases directly as efficient data packages

In [2]:
create_db_package = True
create_dbs = True

### 1.1 Decide on the scenarios to be included for this exercise:
* A business-as-usual scenario, using current trends from output energy system scenarios from OPEN-PROM, EDM-I, FORECAST, and combine without circular economy measures.
* A low-carbon scenario, using low-carbon output energy system scenarios from OPEN-PROM, EDM-I, FORECAST, and combine it to ambitious circular economy measures.

All information is provided in data packages, we import it and define the scenarios below

In [3]:
sps_op = Package(r"datapackage_open_prom.json")
sps_edm_i = Package(r"datapackage_edm_i.json")
sps_forecast = Package(r"datapackage_forecast.json")
sps_itom_por = Package(r"datapackage_itom_por.json")

bau_iam_scenario = 'SSP2-NDC'#"SSP2-PkBudg1000"
bau_iam_scenario = 'SSP2-PkBudg1000' # this would fit better the OCE and CL scenarios for now

#1. OPEN-PROM file with all energy system info
#2. modifying EU steel sector.
#3. Add FORECAST ammonia and methanol markets in Europe.
#3. Integrate PoR in the Netherlands.

# Low-carbon scenario:
scenarios = [
        {"model": "remind", 'year': 2050, "pathway": bau_iam_scenario, 
         "external scenarios": [
                              #{"scenario": "NDC-EI", "data": sps_op}, #Electricity + hydrogen for Europe, OPEN-PROM
                             #{"scenario": "CBAM", "data": sps_edm_i},  #Steel, EDM-invest (now ITOM, EUROPEAN SCOPE for now)
                              {"scenario": "fc_nz", "data": sps_forecast}, #Methanol + Ammonia, FORECAST is focused on low-carbon scenario (Country-specific scope in EU)
                              {"scenario": "OCE", "data": sps_itom_por}, #PoR from ITOM                                                                           
                                   ] },
        {"model": "remind", 'year': 2050, "pathway": bau_iam_scenario, 
         "external scenarios": [
        #                      {"scenario": "NDC-EI", "data": sps_op}, #Electricity for Europe, OPEN-PROM
        #                     {"scenario": "CBAM", "data": sps_edm_i},  #Steel, EDM-invest (now ITOM, EUROPEAN SCOPE for now)
                              {"scenario": "fc_nz", "data": sps_forecast}, #Methanol + Ammonia, FORECAST is focused on low-carbon scenario (Country-specific scope in EU)
                              {"scenario": "CL", "data": sps_itom_por}, #PoR from ITOM                                                                           
                                   ] },

]

In [4]:
# we need to convert ISO3 names to ISO2 for the PoR data using Open-prom, which is in ISO2. We can do this with pycountry.

In [5]:
# Read CSV
df = pd.read_csv("scenario_data/scenario_data_open_prom_init.csv")

def iso3_to_iso2(code):
    try:
        return pycountry.countries.get(alpha_3=code).alpha_2
    except:
        return code  # keep original if not found

df["region"] = df["region"].apply(iso3_to_iso2)

# Save
df.to_csv("scenario_data\scenario_data_open_prom.csv", index=False)

In [6]:
# Create dbs if not existing yet
if DB_NAME_INIT not in bi.databases:
    bi.import_ecoinvent_release("3.12", "cutoff", USER_NAME, USER_PW)
bi.databases

Databases dictionary with 4 object(s):
	ecoinvent-3.12-biosphere
	ecoinvent-3.12-cutoff
	ecoinvent_312_reference
	remind_2050_SSP2-PkBudg1000_fc_nz_OCE

In [7]:
for db in list(bi.databases):
    if db == 'ecoinvent_312_reference':
        pass
        #del bd.databases['ecoinvent_312_reference']
    if 'remind_' in db:
        pass
        #print(f"Deleting {db}...")
        #del bd.databases[db]
list(bi.databases)

['ecoinvent-3.12-biosphere',
 'ecoinvent-3.12-cutoff',
 'ecoinvent_312_reference',
 'remind_2050_SSP2-PkBudg1000_fc_nz_OCE']

In [8]:
# We need to make an additional database with ecoinvent to import LCIs
def generate_reference_database():
    """
    Generate a reference database based on specified parameters.

    This function generates a reference database based on the specified parameters.
    It deletes the existing reference database with the same name if it exists and then creates a new one.

    Returns:
    None

    Example:
    >>> generate_reference_database()
    """

    bd.projects.set_current(PROJECT_NAME)
    pr.clear_cache()
    # Delete old reference database with the same name
    for db_name in list(bi.databases):
        if NAME_REF_DB in db_name:
            print(db_name)
            del bi.databases[db_name]

    # Create a new reference database using NewDatabase
    ndb = pr.NewDatabase(
        scenarios=[{"model": "remind", "pathway": "SSP2-NPi", "year": "2025",
                    "exclude": ["update_electricity", "update_cement", "update_steel", "update_dac",
                                "update_fuels", "update_emissions"]}],
        source_db=DB_NAME_INIT,
        source_version=EI_VERSION,
        key=KEY_PREMISE,
        biosphere_name=BIOSPHERE_DB,
    )

    # Write the new reference database to Brightway2
    ndb.write_db_to_brightway(name=NAME_REF_DB)

if NAME_REF_DB not in bi.databases:
    generate_reference_database()

### 1.4 if we want to export it as a brightway database, we can do this as follows:

In [9]:
def clean(s: str) -> str:
    """Replace spaces with - and ensure it's filename safe."""
    return re.sub(r"\s+", "-", s)

scenario_names = []

for sc in scenarios:
    parts = [sc["model"], str(sc["year"]), sc["pathway"]]
    external = [clean(item["scenario"]) for item in sc["external scenarios"]]
    parts.extend(external)
    scenario_names.append("_".join(parts))
scenario_names

['remind_2050_SSP2-PkBudg1000_fc_nz_OCE',
 'remind_2050_SSP2-PkBudg1000_fc_nz_CL']

In [10]:
if create_dbs:
    from por_supplier_linking import configure_por_export_support, attach_por_supplier_linking
    from por_scenario_inputs import fresh_scenario_definitions
    configure_por_export_support()
    bw_scenarios = fresh_scenario_definitions(scenarios[0:1])
    bw_scenario_names = scenario_names[0:1]

    # Build successfully before replacing the existing scenario databases.
    ndb = pr.NewDatabase(
        scenarios = bw_scenarios,
        source_db=DB_NAME_INIT, # <-- name of the database in the BW2 project. Must be a string.
        source_version=EI_VERSION, # <-- version of ecoinvent. Must be a string.
        key=KEY_PREMISE, # <-- ask the key to run premise from the developers
        biosphere_name=BIOSPHERE_DB)

    attach_por_supplier_linking(ndb, stage="brightway")
    ndb.update()

    # Replace only these export targets, preserving unrelated future databases.
    for db in bw_scenario_names:
        if db in bd.databases:
            print(f"Replacing {db} after successful scenario update")
            del bd.databases[db]

    ndb.write_db_to_brightway(name=bw_scenario_names)

premise v.(2, 5, 2)
+------------------------------------------------------------------+
| Warning                                                          |
+------------------------------------------------------------------+
| Because some of the scenarios can yield LCI databases            |
| containing net negative emission technologies (NET),             |
| it is advised to account for biogenic CO2 flows when calculating |
| Global Warming potential indicators.                             |
| `premise_gwp` provides characterization factors for such flows.  |
| It also provides factors for hydrogen emissions to air.          |
|                                                                  |
| Within your Brightway project:                                   |
| from premise_gwp import add_premise_gwp                          |
| add_premise_gwp()                                                |
+------------------------------------------------------------------+
+-------------

Processing scenarios for all sectors:   0%|     | 0/1 [00:00<?, ?it/s]

Importing c:\Users\terlouw_t\Documents\Projects\TRANSIENCE\lca_transience\inventories/lci-forecast.xlsx
Extracted 1 worksheets in 0.01 seconds
Migration route: 3.10 → 3.11 → 3.12
Applying forward migration 3.10 -> 3.11
Applying forward migration 3.11 -> 3.12
Importing c:\Users\terlouw_t\Documents\Projects\TRANSIENCE\lca_transience\inventories/lci-itom_por.xlsx
Extracted 1 worksheets in 0.06 seconds


Processing scenarios for all sectors: 100%|█| 1/1 [01:18<00:00, 78.68s


Done!

PoR supplier linking 1/1 — 2050 fc_nz + OCE: loading updated inventory
PoR supplier linking 1/1 — 2050 fc_nz + OCE: loading PoR activities and supplier identities
PoR supplier linking 1/1 — 2050 fc_nz + OCE: matching and linking suppliers
PoR supplier linking 1/1 — 2050 fc_nz + OCE: capturing validation baseline
PoR supplier linking 1/1 — 2050 fc_nz + OCE: validating links and cycle changes
PoR supplier linking 1/1 — 2050 fc_nz + OCE: saving linked checkpoint
PoR supplier linking 1/1 — 2050 fc_nz + OCE: ready
PoR supplier linking 1/1 — 2050 fc_nz + OCE: complete (56.7 s; 463 changed exchanges)
PoR supplier linking: 1 scenario(s), 463 changed exchange(s). Reports: C:\Users\terlouw_t\Documents\Projects\TRANSIENCE\lca_transience\export\por_supplier_linking
Replacing remind_2050_SSP2-PkBudg1000_fc_nz_OCE after successful scenario update
13:26:17+0200 [warning  ] Removing database node(s) from calculation setup polymers
13:27:09+0200 [info     ] Vacuuming database            
Write n

Brightway database written: remind_2050_SSP2-PkBudg1000_fc_nz_OCE
Generate scenario report.
Report saved under c:\Users\terlouw_t\Documents\Projects\TRANSIENCE\lca_transience\export\scenario_report.
Generate change report.


### 2. Export the defined scenarios for chosen years as data packages needed for PATHWAYS in the next step (see the other Notebook `2_calc_impacts_ALL.ipynb`)

scenarios

In [11]:
if create_db_package:
    import importlib, sys
    import premise_compat as por_compat
    import inventory_supplier_compat as por_inventory_compat
    import por_supplier_linking as por_linking
    import por_scenario_inputs as por_inputs
    # Ordinary imports keep old helper functions in a long-lived kernel.
    importlib.reload(por_compat)
    importlib.reload(por_inventory_compat)
    importlib.reload(por_linking)
    importlib.reload(por_inputs)

    print(f"Running premise {getattr(pr, '__version__', 'unknown')} from {getattr(pr, '__file__', 'unknown')}; Python: {sys.executable}", flush=True)

    por_linking.configure_por_export_support()
    package_scenarios = por_inputs.fresh_scenario_definitions(scenarios)
    print("Preparing fresh inputs for the six-year Pathways package...", flush=True)

    #if 'external scenarios' in str(scenario):
    ndb = pr.PathwaysDataPackage(
        scenarios=package_scenarios,
        #years=[2025, 2050],
        years=[2025, 2030, 2035, 2040, 2045, 2050],#, 2050],
        source_db=DB_NAME_INIT,
        source_version=EI_VERSION,
        key=KEY_PREMISE,
        biosphere_name=BIOSPHERE_DB )

    # The package calls its internal NewDatabase.update() before exporting matrices.
    print("Package builder ready; attaching the PoR post-update linking step.", flush=True)
    por_linking.attach_por_pathways_export(ndb, stage="pathways")
    
    ndb.create_datapackage(
        name=f"remind-transience_por",
        contributors=[
            {"name": "Tom Terlouw",
            "email": "tom.terlouw@psi.ch"}] )

Running premise (2, 5, 2) from c:\Users\terlouw_t\.conda\envs\premise_pathways_new\Lib\site-packages\premise\__init__.py; Python: c:\Users\terlouw_t\.conda\envs\premise_pathways_new\python.exe
Preparing fresh inputs for the six-year Pathways package...
premise v.(2, 5, 2)
+------------------------------------------------------------------+
| Warning                                                          |
+------------------------------------------------------------------+
| Because some of the scenarios can yield LCI databases            |
| containing net negative emission technologies (NET),             |
| it is advised to account for biogenic CO2 flows when calculating |
| Global Warming potential indicators.                             |
| `premise_gwp` provides characterization factors for such flows.  |
| It also provides factors for hydrogen emissions to air.          |
|                                                                  |
| Within your Brightway project:     

Processing scenarios for all sectors:   0%|    | 0/12 [00:00<?, ?it/s]

Importing c:\Users\terlouw_t\Documents\Projects\TRANSIENCE\lca_transience\inventories/lci-forecast.xlsx
Extracted 1 worksheets in 0.01 seconds
Migration route: 3.10 → 3.11 → 3.12
Applying forward migration 3.10 -> 3.11
Applying forward migration 3.11 -> 3.12
Importing c:\Users\terlouw_t\Documents\Projects\TRANSIENCE\lca_transience\inventories/lci-itom_por.xlsx
Extracted 1 worksheets in 0.07 seconds
No suppliers found for market for toluene (por) in NL. No market created. This may cause linking issue.
No suppliers found for market for methanol (por) in NL. No market created. This may cause linking issue.


Processing scenarios for all sectors:   8%| | 1/12 [01:18<14:27, 78.84

Importing c:\Users\terlouw_t\Documents\Projects\TRANSIENCE\lca_transience\inventories/lci-forecast.xlsx
Extracted 1 worksheets in 0.01 seconds
Migration route: 3.10 → 3.11 → 3.12
Applying forward migration 3.10 -> 3.11
Applying forward migration 3.11 -> 3.12
Importing c:\Users\terlouw_t\Documents\Projects\TRANSIENCE\lca_transience\inventories/lci-itom_por.xlsx
Extracted 1 worksheets in 0.06 seconds
No suppliers found for market for toluene (por) in NL. No market created. This may cause linking issue.
No suppliers found for market for methanol (por) in NL. No market created. This may cause linking issue.


...

Importing c:\Users\terlouw_t\Documents\Projects\TRANSIENCE\lca_transience\inventories/lci-forecast.xlsx
Extracted 1 worksheets in 0.01 seconds
Migration route: 3.10 → 3.11 → 3.12
Applying forward migration 3.10 -> 3.11
Applying forward migration 3.11 -> 3.12
Importing c:\Users\terlouw_t\Documents\Projects\TRANSIENCE\lca_transience\inventories/lci-itom_por.xlsx
Extracted 1 worksheets in 0.05 seconds


Processing scenarios for all sectors:  58%|▌| 7/12 [08:46<06:09, 73.93

Importing c:\Users\terlouw_t\Documents\Projects\TRANSIENCE\lca_transience\inventories/lci-forecast.xlsx
Extracted 1 worksheets in 0.01 seconds
Migration route: 3.10 → 3.11 → 3.12
Applying forward migration 3.10 -> 3.11
Applying forward migration 3.11 -> 3.12
Importing c:\Users\terlouw_t\Documents\Projects\TRANSIENCE\lca_transience\inventories/lci-itom_por.xlsx
Extracted 1 worksheets in 0.06 seconds


Processing scenarios for all sectors:  67%|▋| 8/12 [10:44<05:22, 80.54


KeyboardInterrupt: 

### 3. Continue in Notebook `2_calc_impacts.ipynb` to calculate environmental burdens of sectors or entire energy system scenarios using the exported data packages from the previous step